# 第2章: UNIXコマンド

[言語処理100本ノック 2025 第2章](https://nlp100.github.io/2025/ja/ch02.html)（岡崎直観）を参照。最初のコードセルで配布データを取得し、以降の各問でPythonの解答をUNIXコマンドと照合します。

In [1]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
import os
import random
import subprocess

path = Path("popular-names.txt")
if not path.exists():
    urlretrieve(
        "https://raw.githubusercontent.com/hisamoto-lang/python100-exercise-drill/chapter02/data/popular-names.txt",
        path,
    )

lines = path.read_text(encoding="utf-8").splitlines()
names, genders, counts, years = zip(*(line.split("\t") for line in lines))
N = 10

def unix(command, *args, cwd=None):
    return subprocess.check_output(
        ["bash", "-o", "pipefail", "-c", command, "bash", *map(str, args)],
        text=True,
        encoding="utf-8",
        env={**os.environ, "LC_ALL": "C"},
        cwd=cwd,
    )

## 10. 行数のカウント
入力ファイルの行数を数え、wcで確認する。

In [2]:
line_count = len(lines)
print(line_count)

# UNIXコマンドで確認
assert line_count == int(unix('wc -l < "$1"', path))
print("wc と一致")

2780
wc と一致


## 11. 先頭からN行を出力
先頭のN行（ここでは10行）を表示し、headで確認する。

In [3]:
first = lines[:N]
print(*first, sep="\n")

# UNIXコマンドで確認
assert first == unix('head -n "$2" "$1"', path, N).splitlines()
print("head と一致")

Mary	F	7065	1880
Anna	F	2604	1880
Emma	F	2003	1880
Elizabeth	F	1939	1880
Minnie	F	1746	1880
Margaret	F	1578	1880
Ida	F	1472	1880
Alice	F	1414	1880
Bertha	F	1320	1880
Sarah	F	1288	1880
head と一致


## 12. 末尾のN行を出力
末尾のN行（ここでは10行）を表示し、tailで確認する。

In [4]:
last = lines[-N:] if N else []
print(*last, sep="\n")

# UNIXコマンドで確認
assert last == unix('tail -n "$2" "$1"', path, N).splitlines()
print("tail と一致")

Liam	M	19837	2018
Noah	M	18267	2018
William	M	14516	2018
James	M	13525	2018
Oliver	M	13389	2018
Benjamin	M	13381	2018
Elijah	M	12886	2018
Lucas	M	12585	2018
Mason	M	12435	2018
Logan	M	12352	2018
tail と一致


## 13. タブをスペースに置換
先頭10行にある各タブをスペース1文字に置き換え、trで確認する。

In [5]:
converted = [line.replace("\t", " ") for line in lines[:10]]
print(*converted, sep="\n")

# UNIXコマンドで確認
assert converted == unix(r'head -n 10 "$1" | tr "\t" " "', path).splitlines()
print("head と tr の結果に一致")

Mary F 7065 1880
Anna F 2604 1880
Emma F 2003 1880
Elizabeth F 1939 1880
Minnie F 1746 1880
Margaret F 1578 1880
Ida F 1472 1880
Alice F 1414 1880
Bertha F 1320 1880
Sarah F 1288 1880
head と tr の結果に一致


## 14. 1列目を出力
先頭10行の1列目を表示し、cutで確認する。

In [6]:
first_names = list(names[:10])
print(*first_names, sep="\n")

# UNIXコマンドで確認
assert first_names == unix('head -n 10 "$1" | cut -f1', path).splitlines()
print("head と cut の結果に一致")

Mary
Anna
Emma
Elizabeth
Minnie
Margaret
Ida
Alice
Bertha
Sarah
head と cut の結果に一致


## 15. ファイルをN分割する
入力を行単位でN個（ここでは10個）のファイルに分割し、splitで確認する。

In [7]:
# 元の順番を保ち、行数の差を最大1行にする。
output_dir = Path("ch02_parts")
output_dir.mkdir(exist_ok=True)
size, extra = divmod(len(lines), N)
parts = []
offset = 0
for index in range(N):
    stop = offset + size + (index < extra)
    part = output_dir / f"part_{index:02d}.txt"
    part.write_text("".join(line + "\n" for line in lines[offset:stop]), encoding="utf-8")
    parts.append(part)
    offset = stop
    print(part, size + (index < extra))

# UNIXコマンドで確認
# GNU split はバイト数を基準に境界を決めるため、件数と結合結果を比較する。
with TemporaryDirectory() as temp_dir:
    unix('split -n "l/$2" -d "$1" part_', path.resolve(), N, cwd=temp_dir)
    unix_parts = sorted(Path(temp_dir).glob("part_*"))
    assert len(unix_parts) == len(parts)
    def joined_lines(files):
        return [line for file in files for line in file.read_text(encoding="utf-8").splitlines()]

    assert joined_lines(parts) == joined_lines(unix_parts) == lines
print("split と分割数・結合結果が一致")

ch02_parts\part_00.txt 278
ch02_parts\part_01.txt 278
ch02_parts\part_02.txt 278
ch02_parts\part_03.txt 278
ch02_parts\part_04.txt 278
ch02_parts\part_05.txt 278
ch02_parts\part_06.txt 278
ch02_parts\part_07.txt 278
ch02_parts\part_08.txt 278
ch02_parts\part_09.txt 278
split と分割数・結合結果が一致


## 16. ランダムに各行を並び替える
行の内容を保ってランダムに並べ替え、shufの結果と行の内訳を比較する。

In [8]:
shuffled = lines.copy()
random.shuffle(shuffled)
print(*shuffled[:10], sep="\n")

# UNIXコマンドで確認
unix_shuffled = unix('shuf "$1"', path).splitlines()
assert Counter(shuffled) == Counter(unix_shuffled) == Counter(lines)
print("shuf と行の内容・出現回数が一致")

Susan	F	46571	1956
Michael	M	81021	1965
Emma	F	2872	1896
Mary	F	14486	1902
Margaret	F	26549	1924
Steven	M	38397	1956
Sandra	F	27667	1951
Stephanie	F	22330	1983
William	M	6904	1907
Samantha	F	25645	1991
shuf と行の内容・出現回数が一致


## 17. 1列目の文字列の異なり
1列目に現れる名前の一覧を作り、cut・sort・uniqで確認する。

In [9]:
unique_names = sorted(set(names))
print(*unique_names, sep="\n")

# UNIXコマンドで確認
assert unique_names == unix('cut -f1 "$1" | sort | uniq', path).splitlines()
print("cut、sort、uniq の結果に一致")

Abigail
Aiden
Alexander
Alexis
Alice
Amanda
Amelia
Amy
Andrew
Angela
Anna
Annie
Anthony
Ashley
Austin
Ava
Barbara
Benjamin
Bertha
Bessie
Betty
Brandon
Brian
Brittany
Carol
Carolyn
Charles
Charlotte
Chloe
Christopher
Clara
Crystal
Cynthia
Daniel
David
Deborah
Debra
Donald
Donna
Doris
Dorothy
Edward
Elijah
Elizabeth
Emily
Emma
Ethan
Ethel
Evelyn
Florence
Frances
Frank
Gary
George
Hannah
Harper
Harry
Heather
Helen
Henry
Ida
Isabella
Jacob
James
Jason
Jayden
Jeffrey
Jennifer
Jessica
Joan
John
Joseph
Joshua
Judith
Julie
Justin
Karen
Kathleen
Kelly
Kimberly
Larry
Laura
Lauren
Liam
Lillian
Linda
Lisa
Logan
Lori
Lucas
Madison
Margaret
Marie
Mark
Mary
Mason
Matthew
Megan
Melissa
Mia
Michael
Michelle
Mildred
Minnie
Nancy
Nicholas
Nicole
Noah
Oliver
Olivia
Pamela
Patricia
Rachel
Rebecca
Richard
Robert
Ronald
Ruth
Samantha
Sandra
Sarah
Scott
Sharon
Shirley
Sophia
Stephanie
Steven
Susan
Tammy
Taylor
Thomas
Tracy
Tyler
Virginia
Walter
William
cut、sort、uniq の結果に一致


## 18. 1列目の文字列の出現頻度
名前の出現回数を数え、多い順に表示する。cut・uniq・sortで確認する。

In [10]:
# 同じ頻度の名前は名前順。
frequencies = sorted(Counter(names).items(), key=lambda item: (-item[1], item[0]))
for name, count in frequencies:
    print(count, name)

# UNIXコマンドで確認
frequency_rows = unix('cut -f1 "$1" | sort | uniq -c | sort -k1,1nr -k2,2', path)
frequency_pairs = []
for row in frequency_rows.splitlines():
    count, name = row.split()
    frequency_pairs.append((name, int(count)))
assert frequencies == frequency_pairs
print("cut、uniq、sort の結果に一致")

118 James
111 William
108 John
108 Robert
92 Mary
75 Charles
74 Michael
73 Elizabeth
70 Joseph
60 Margaret
58 George
58 Thomas
57 David
51 Richard
45 Helen
43 Christopher
43 Frank
41 Anna
40 Edward
39 Ruth
38 Patricia
37 Matthew
36 Dorothy
35 Emma
32 Barbara
31 Daniel
31 Joshua
26 Emily
26 Jennifer
26 Linda
26 Sarah
25 Jacob
25 Jessica
24 Betty
24 Mildred
24 Susan
23 Ashley
23 Henry
22 Nancy
21 Andrew
20 Amanda
20 Donald
20 Florence
20 Marie
19 Samantha
18 Karen
18 Lisa
18 Madison
18 Melissa
18 Olivia
17 Abigail
17 Stephanie
16 Ethel
16 Mark
16 Sandra
15 Angela
15 Carol
15 Ethan
15 Frances
15 Heather
15 Isabella
15 Michelle
14 Amy
14 Ava
14 Kimberly
14 Shirley
13 Brian
13 Deborah
13 Hannah
13 Jason
13 Nicole
13 Sophia
13 Virginia
12 Bertha
12 Donna
12 Minnie
11 Cynthia
10 Alice
10 Brittany
10 Doris
10 Mia
10 Nicholas
10 Noah
10 Ronald
9 Debra
9 Joan
9 Tyler
8 Alexander
8 Alexis
8 Clara
8 Ida
8 Judith
8 Mason
8 Taylor
7 Brandon
7 Harry
7 Liam
7 Sharon
7 Steven
7 Tammy
6 Anthony
5 Annie


## 19. 3列目の数値の降順に各行を並び替える
行の内容を保ち、3列目を数値として降順に並べ、sortで確認する。

In [11]:
# 同じ人数の行は入力順を保つ。
ranked = sorted(lines, key=lambda line: int(line.split("\t")[2]), reverse=True)
print(*ranked[:10], sep="\n")

# UNIXコマンドで確認
assert ranked == unix("""sort -s -t "$(printf '\t')" -k3,3nr "$1" """, path).splitlines()
print("sort と全行が一致")

Linda	F	99689	1947
Linda	F	96211	1948
James	M	94757	1947
Michael	M	92704	1957
Robert	M	91640	1947
Linda	F	91016	1949
Michael	M	90656	1956
Michael	M	90517	1958
James	M	88584	1948
Michael	M	88528	1954
sort と全行が一致
